In [6]:
import pandas as pd
import numpy as np

final = pd.read_csv(
    "../data/lodhi_hourly_with_weather.csv",
    index_col=0,
    parse_dates=True
)

print(final.shape)

(17544, 13)


In [7]:
pm = final["PM2.5 (ug/m3)"]
feat = pd.DataFrame(index=final.index)

# pichhle ghanto ka pm2.5
feat["pm_now"] = pm
for l in [1, 2, 3, 6, 12, 24, 48, 72]:
    feat[f"pm_lag_{l}"] = pm.shift(l)

# rolling average
feat["pm_roll_24"] = pm.rolling(24).mean()
feat["pm_roll_72"] = pm.rolling(72).mean()

In [8]:
# abhi ka weather
for c in ["temperature_2m", "relative_humidity_2m", "wind_speed_10m",
          "wind_direction_10m", "surface_pressure", "precipitation"]:
    feat[c] = final[c]

# time ke features
feat["hour"] = final.index.hour
feat["month"] = final.index.month
feat["dayofweek"] = final.index.dayofweek

In [9]:
# 24 ghante baad ka pm2.5 predict karna hai
horizon = 24
data = feat.copy()
data["target"] = pm.shift(-horizon)
data = data.dropna()
print(data.shape)

# time ke hisaab se split, random nahi
train = data[data.index < "2025-09-27"]
test = data[data.index >= "2025-09-27"]
print(train.shape, test.shape)

X_train, y_train = train.drop(columns="target"), train["target"]
X_test, y_test = test.drop(columns="target"), test["target"]

(15147, 21)
(13021, 21) (2126, 21)


In [11]:
import platform
import xgboost

print("Architecture:", platform.machine())
print("XGBoost:", xgboost.__version__)

Architecture: arm64
XGBoost: 3.4.1


In [12]:
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

model = XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=5,
                     subsample=0.8, colsample_bytree=0.8, random_state=42)
model.fit(X_train, y_train)
pred = model.predict(X_test)

# persistence baseline: kal bhi pm2.5 aaj jaisa hi rahega
pers = X_test["pm_now"]

def rmse(a, b):
    return np.sqrt(mean_squared_error(a, b))

print("xgboost   mae:", round(mean_absolute_error(y_test, pred), 2), " rmse:", round(rmse(y_test, pred), 2))
print("persistence mae:", round(mean_absolute_error(y_test, pers), 2), " rmse:", round(rmse(y_test, pers), 2))

xgboost   mae: 47.15  rmse: 76.21
persistence mae: 47.64  rmse: 82.46
